![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Build the Core

*Project 3 — AI Product: From Model to Product*

---

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## 0. Setup

In [1]:
import sys
from pathlib import Path
import json
import time

import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_colwidth", 200)

In [2]:
# Load job listings
df = pd.read_csv(ROOT / "data/raw/jobs.csv")

# Prepare job texts
df["job_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("")
)

# Load candidate profiles
with open(ROOT / "data/candidate_profiles.json", encoding="utf-8") as f:
    candidate_profiles = json.load(f)

# Example candidate for comparing recommendations
candidate_text = candidate_profiles["ml_engineer"]

print("Jobs:", len(df))
print("Candidate profiles:", len(candidate_profiles))

Jobs: 162
Candidate profiles: 5


---
## 1. Model track — compare at least three approaches

On the same folds, or the same validation set, against the baseline from notebook 01. For example:

1. TF-IDF with a linear model (your baseline), tuned.
2. Sentence embeddings (for example `all-MiniLM-L6-v2`) with a classifier.
3. A fine-tuned transformer (DistilBERT on a sample, as in Week 7) or a zero-shot model.

Report the mean and the spread, or the validation score, and the **training time** of each. Track the runs with MLflow if you can.

In [3]:
# %pip install transformers sentencepiece

### TF-IDF baseline

In [4]:
# TF-IDF baseline
documents = [candidate_text] + df["job_text"].tolist()

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

df["tfidf_score"] = cosine_similarity(
    tfidf_matrix[:1],
    tfidf_matrix[1:]
).flatten()

display(
    df.nlargest(5, "tfidf_score")[
        ["title", "tfidf_score"]
    ]
)

,title,tfidf_score
44,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€",0.219602
40,(Junior) Machine Learning Engineer (m/w/d),0.186809
25,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",0.174439
76,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d,0.156421
45,Senior Machine Learning Engineer – Time Series & Forecasting (w/m/d),0.135942


### Approach 2 — Sentence Transformers

The second approach uses a pretrained multilingual Sentence Transformer (`paraphrase-multilingual-MiniLM-L12-v2`).

Unlike TF-IDF, which relies on shared vocabulary, Sentence Transformers creates dense vector representations that capture semantic relationships between texts.

I encode the candidate profile and job descriptions, calculate cosine similarity, and rank the jobs according to their similarity scores.

The goal is to identify relevant positions even when candidates and job advertisements use different terminology.

In [5]:
# Load pretrained embedding model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode candidate and job descriptions
candidate_embedding = model.encode([candidate_text])

job_embeddings = model.encode(
    df["job_text"].tolist(),
    show_progress_bar=True
)

# Calculate semantic similarity
semantic_scores = cosine_similarity(
    candidate_embedding,
    job_embeddings
).flatten()

df["semantic_score"] = semantic_scores

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [6]:
comparison = pd.DataFrame({
    "TF-IDF": df.nlargest(5, "tfidf_score")["title"].tolist(),
    "Sentence Transformers": df.nlargest(5, "semantic_score")["title"].tolist()
})

display(comparison)

,TF-IDF,Sentence Transformers
0,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€","(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur"
1,(Junior) Machine Learning Engineer (m/w/d),(Junior) Machine Learning Engineer (m/w/d)
2,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d
3,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€"
4,Senior Machine Learning Engineer – Time Series & Forecasting (w/m/d),Senior Machine Learning Engineer – Time Series & Forecasting (w/m/d)


In [7]:
comparison.to_csv(
    "../data/model_comparison_2.csv",
    index=False
)

### Approach 3 — Zero-Shot Classification

The third approach uses a pretrained multilingual zero-shot classification model (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`).

For each candidate–job pair, the model estimates probabilities for three categories: strong match, partial match, and poor match.

I convert these probabilities into a weighted relevance score between 0 and 1.

The experiment required approximately 36 minutes on CPU to score 140 candidate–job combinations. Because of this computational cost, I saved the predictions to a CSV file for reuse during evaluation.

This approach was included to investigate whether a more complex language model could produce better matches than cosine similarity.

### !!! Zero-shot inference example (optional — computationally expensive) !!!

In [8]:
# start = time.perf_counter()

# zero_shot = pipeline(
#     "zero-shot-classification",
#     model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
#     device=-1  # Use CPU
# )

# loading_time = time.perf_counter() - start

# print(f"Model loading time: {loading_time:.2f} seconds")

In [9]:
# candidate = """
# Junior machine learning engineer with Python,
# Pandas, scikit-learn, SQL and predictive modeling experience.
# """

# job = """
# Junior Machine Learning Engineer.
# Develop machine learning models using Python,
# SQL and scikit-learn.
# """

# # Describe both the candidate and the vacancy
# text = f"""
# Candidate experience:
# {candidate}

# Job requirements:
# {job}
# """

# labels = [
#     "a strong match between candidate and job",
#     "a partial match between candidate and job",
#     "a poor match between candidate and job"
# ]

# start = time.perf_counter()

# result = zero_shot(
#     text,
#     candidate_labels=labels,
#     hypothesis_template="This candidate-job comparison represents {}.",
#     multi_label=False
# )

# inference_time = time.perf_counter() - start

# for label, score in zip(result["labels"], result["scores"]):
#     print(f"{label}: {score:.3f}")

# print(f"\nInference time: {inference_time:.2f} seconds")

In [10]:
# def zero_shot_match(candidate_text, job_text):
#     """
#     Evaluate how relevant a job is for a candidate.

#     Returns a relevance score between 0 and 1.
#     """

#     text = f"""
#     Candidate experience:
#     {candidate_text}

#     Job description:
#     {job_text}
#     """

#     labels = [
#         "a strong match between candidate and job",
#         "a partial match between candidate and job",
#         "a poor match between candidate and job"
#     ]

#     result = zero_shot(
#         text,
#         candidate_labels=labels,
#         hypothesis_template="This candidate-job comparison represents {}.",
#         multi_label=False,
#         truncation=True
#     )

#     # Convert results into a dictionary
#     scores = dict(zip(result["labels"], result["scores"]))

#     # Weighted relevance score
#     relevance_score = (
#         scores[labels[0]] * 1.0
#         + scores[labels[1]] * 0.5
#         + scores[labels[2]] * 0.0
#     )

#     return relevance_score

In [11]:
# Load previously generated zero-shot scores
zero_shot_results = pd.read_csv(
    ROOT / "data/zero_shot_scores.csv",
    dtype={"job_id": str}
)

print("Saved candidate-job predictions:", len(zero_shot_results))

display(zero_shot_results.head())

Saved candidate-job predictions: 140


,profile_id,job_id,zero_shot_score
0,ml_engineer,5953648768,0.613409
1,ml_engineer,5943389711,0.672074
2,ml_engineer,5956672032,0.510513
3,ml_engineer,5939347586,0.621427
4,ml_engineer,5892995304,0.626463


In [12]:
# jobs = pd.read_csv(
#     "../data/evaluation_jobs.csv",
#     dtype={"id": str}
# )

# with open("../data/candidate_profiles.json", encoding="utf-8") as f:
#     candidate_profiles = json.load(f)

# candidate = candidate_profiles["ml_engineer"]

# results = []

# start = time.perf_counter()

# for _, job in jobs.head(5).iterrows():

#     job_text = f"{job['title']}. {job['description']}"

#     score = zero_shot_match(candidate, job_text)

#     results.append({
#         "title": job["title"],
#         "zero_shot_score": score
#     })

# elapsed = time.perf_counter() - start

# test_results = pd.DataFrame(results)

# display(
#     test_results.sort_values(
#         "zero_shot_score",
#         ascending=False
#     )
# )

# print(f"Total inference time: {elapsed:.2f} seconds")

In [13]:
# output = Path("../data/zero_shot_scores.csv")

# # Load existing progress if available
# if output.exists():
#     results_df = pd.read_csv(
#         output,
#         dtype={"job_id": str}
#     )
#     results = results_df.to_dict("records")
# else:
#     results = []

# # Track already processed pairs
# completed = {
#     (row["profile_id"], str(row["job_id"]))
#     for row in results
# }

# start = time.perf_counter()

# for profile_id, candidate_text in candidate_profiles.items():

#     for _, job in jobs.iterrows():

#         job_id = str(job["id"])
#         pair = (profile_id, job_id)

#         # Skip previously processed pairs
#         if pair in completed:
#             continue

#         job_text = f"{job['title']}. {job['description']}"

#         score = zero_shot_match(
#             candidate_text,
#             job_text
#         )

#         results.append({
#             "profile_id": profile_id,
#             "job_id": job_id,
#             "zero_shot_score": score
#         })

#         completed.add(pair)

#         # Save progress after every prediction
#         pd.DataFrame(results).to_csv(
#             output,
#             index=False
#         )

#     print(f"Completed: {profile_id}")

# elapsed = time.perf_counter() - start

# print(f"\nTotal predictions saved: {len(results)}")
# print(f"Execution time this run: {elapsed:.2f} seconds")

In [14]:
# Load previously calculated zero-shot predictions
zero_shot_results = pd.read_csv(
    ROOT / "data/zero_shot_scores.csv",
    dtype={"job_id": str}
)

# Display results
print("Saved predictions:", len(zero_shot_results))
print("Unique candidates:", zero_shot_results["profile_id"].nunique())
print("Unique jobs:", zero_shot_results["job_id"].nunique())

display(zero_shot_results.head())

Saved predictions: 140
Unique candidates: 5
Unique jobs: 28


,profile_id,job_id,zero_shot_score
0,ml_engineer,5953648768,0.613409
1,ml_engineer,5943389711,0.672074
2,ml_engineer,5956672032,0.510513
3,ml_engineer,5939347586,0.621427
4,ml_engineer,5892995304,0.626463


---
## 2. Model track — tune and choose



### Model Comparison and Selection

I compared three approaches using the same five candidate profiles and fixed evaluation set of 28 job advertisements.

The results were:

| Approach | Precision@5 | nDCG@5 |
|---|---:|---:|
| TF-IDF + cosine similarity | 0.24 | 0.675 |
| Sentence Transformers + cosine similarity | **0.44** | **0.781** |
| Zero-shot classification | 0.08 | 0.242 |

Sentence Transformers achieved the best performance on both metrics.

Compared with TF-IDF, it improved nDCG@5 by approximately 15.7% and Precision@5 by 0.20.

Zero-shot classification performed considerably worse and was also much slower.

I did not perform additional hyperparameter tuning because of the limited project timeframe. Instead, I selected the best-performing approach from the three implemented methods.

These results are exploratory because the evaluation set is small and relevance labels were revised with AI assistance.

---
## 5. The final version

Which version you will evaluate in notebook 03 and use in the app, and why. Save it: the fitted pipeline with `joblib` (model track), or the index and the configuration (LLM track), in `models/`.

### Final Model Selection

I selected **Sentence Transformers (`paraphrase-multilingual-MiniLM-L12-v2`) with cosine similarity** as the final matching approach.

The decision was based on three factors:

- **Recommendation quality:** It achieved the highest Precision@5 and nDCG@5.
- **Practical performance:** Generating embeddings is more efficient than running zero-shot classification for every candidate–job pair.
- **Integration:** The pretrained model can be loaded directly in Streamlit and reused for matching new candidate profiles.

The final application allows users to enter a description of their experience or upload a PDF CV, filter job listings by location and work arrangement, and receive ranked recommendations.

The model is loaded using `src/functions.py`, while Streamlit handles the user interface.

The main limitation is that semantic similarity does not necessarily guarantee that a candidate meets a job's requirements. Future improvements could include structured skill matching, seniority filtering, and better vacancy data.